In [44]:
import pandas as pd

In [48]:
import pandas as pd

df = pd.read_csv(
    "food_waste_management_dataset_cleaned.csv"
)

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 300
Columns: 18


In [49]:
print(
    "Invalid consumed records:",
    (df['food_consumed_portions'] > df['actual_students']).sum()
)

print(
    "Invalid prepared records:",
    (df['food_consumed_portions'] > df['food_prepared_portions']).sum()
)

Invalid consumed records: 0
Invalid prepared records: 0


In [50]:
# OVERALL FOOD WASTE ANALYSIS 

print("Total records:", len(df))

print("Average food prepared:",
      round(df['food_prepared_portions'].mean(), 2))

print("Average food consumed:",
      round(df['food_consumed_portions'].mean(), 2))

print("Average food wasted:",
      round(df['food_wasted_portions'].mean(), 2))

print("Average waste percentage:",
      round(df['waste_percentage'].mean(), 2), "%")

print("Total food wasted:",
      df['food_wasted_portions'].sum())

Total records: 300
Average food prepared: 403.07
Average food consumed: 340.32
Average food wasted: 62.75
Average waste percentage: 15.83 %
Total food wasted: 18824


In [51]:
#  MENU ANALYSIS

menu_analysis = (
    df.groupby('menu')
    .agg(
        avg_prepared=('food_prepared_portions', 'mean'),
        avg_consumed=('food_consumed_portions', 'mean'),
        avg_wasted=('food_wasted_portions', 'mean'),
        avg_waste_percentage=('waste_percentage', 'mean')
    )
    .sort_values('avg_waste_percentage', ascending=False)
)

print(menu_analysis.round(2))

               avg_prepared  avg_consumed  avg_wasted  avg_waste_percentage
menu                                                                       
Khichdi              441.83        339.93      101.90                 23.18
Poha                 400.56        319.64       80.92                 20.94
Veg Pulao            399.49        327.11       72.38                 18.53
Rajma Rice           408.85        342.27       66.58                 17.02
Idli Sambar          386.50        321.20       65.30                 16.59
Dal Rice             414.03        357.36       56.67                 13.63
Dal Roti             392.79        342.11       50.68                 13.55
Chole Bhature        385.55        341.69       43.86                 11.84
Paneer Rice          364.63        323.63       41.00                 11.62
Pav Bhaji            430.24        392.76       37.48                  8.65


In [ ]:
#  each day food wastage in percentage

day_analysis = (
    df.groupby('day_of_week')
    .agg(
        avg_prepared=('food_prepared_portions', 'mean'),
        avg_consumed=('food_consumed_portions', 'mean'),
        avg_wasted=('food_wasted_portions', 'mean'),
        avg_waste_percentage=('waste_percentage', 'mean')
    )
)

print(day_analysis.round(2))

             avg_prepared  avg_consumed  avg_wasted  avg_waste_percentage
day_of_week                                                              
Friday             497.98        422.86       75.12                 15.15
Monday             419.51        356.51       63.00                 15.28
Saturday           336.14        282.53       53.60                 15.84
Sunday             208.09        171.58       36.51                 17.62
Thursday           439.56        369.60       69.95                 16.11
Tuesday            474.12        404.65       69.47                 15.00
Wednesday          447.12        375.33       71.79                 15.81


In [ ]:
#  WHEATHER ANALYSIS

weather_analysis = (
    df.groupby("weather")
    .agg(
        avg_students=("actual_students", "mean"),
        avg_consumed=("food_consumed_portions", "mean"),
        avg_wasted=("food_wasted_portions", "mean"),
        avg_waste_percentage=("waste_percentage", "mean")
    )
    .sort_values("avg_waste_percentage", ascending=False)
)

print(weather_analysis.round(2))

         avg_students  avg_consumed  avg_wasted  avg_waste_percentage
weather                                                              
Hot            389.63        331.00       74.67                 18.59
Cloudy         373.75        333.31       59.31                 16.31
Normal         388.57        346.29       59.26                 14.85
Cool           396.29        347.33       63.43                 14.64
Rainy          374.93        338.52       56.36                 14.60


In [55]:
# TEMPERATURE ANALYSIS

temperature_corr = df[
    [
        "temperature_c",
        "food_consumed_portions",
        "food_wasted_portions",
        "waste_percentage"
    ]
].corr()

print(temperature_corr.round(2))

                        temperature_c  food_consumed_portions  \
temperature_c                    1.00                   -0.04   
food_consumed_portions          -0.04                    1.00   
food_wasted_portions             0.11                    0.45   
waste_percentage                 0.16                   -0.31   

                        food_wasted_portions  waste_percentage  
temperature_c                           0.11              0.16  
food_consumed_portions                  0.45             -0.31  
food_wasted_portions                    1.00              0.63  
waste_percentage                        0.63              1.00  


In [56]:
menu_attendance = (
    df.groupby("menu")
    .agg(
        avg_students=("actual_students", "mean"),
        avg_prepared=("food_prepared_portions", "mean"),
        avg_consumed=("food_consumed_portions", "mean"),
        avg_wasted=("food_wasted_portions", "mean"),
        avg_waste_percentage=("waste_percentage", "mean")
    )
)

print(menu_attendance.round(2))

               avg_students  avg_prepared  avg_consumed  avg_wasted  \
menu                                                                  
Chole Bhature        368.03        385.55        341.69       43.86   
Dal Rice             398.69        414.03        357.36       56.67   
Dal Roti             377.71        392.79        342.11       50.68   
Idli Sambar          370.95        386.50        321.20       65.30   
Khichdi              428.90        441.83        339.93      101.90   
Paneer Rice          348.44        364.63        323.63       41.00   
Pav Bhaji            415.08        430.24        392.76       37.48   
Poha                 381.42        400.56        319.64       80.92   
Rajma Rice           392.12        408.85        342.27       66.58   
Veg Pulao            374.19        399.49        327.11       72.38   

               avg_waste_percentage  
menu                                 
Chole Bhature                 11.84  
Dal Rice                      13.

In [ ]:

#  USING LINEAR REGESSION MODEL

import pandas as pd
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ==========================================
# 1. LOAD CLEANED DATASET
# ==========================================

df = pd.read_csv(
    "food_waste_management_dataset_cleaned.csv"
)

print("Dataset loaded:", df.shape)


# ==========================================
# 2. SELECT FEATURES
# ==========================================

features = [
    "expected_students",
    "menu",
    "day_of_week",
    "weather",
    "temperature_c",
    "is_holiday",
    "is_exam_day",
    "special_event"
]

target = "food_consumed_portions"

X = df[features]
y = df[target]


# ==========================================
# 3. IDENTIFY COLUMN TYPES
# ==========================================

categorical_features = [
    "menu",
    "day_of_week",
    "weather"
]

numerical_features = [
    "expected_students",
    "temperature_c",
    "is_holiday",
    "is_exam_day",
    "special_event"
]


# ==========================================
# 4. PREPROCESSING
# ==========================================

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


# ==========================================
# 5. CREATE MODEL PIPELINE
# ==========================================

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)


# ==========================================
# 6. TRAIN / TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))


# ==========================================
# 7. TRAIN MODEL
# ==========================================

model.fit(X_train, y_train)


# ==========================================
# 8. MAKE PREDICTIONS
# ==========================================

y_pred = model.predict(X_test)


# ==========================================
# 9. EVALUATE MODEL
# ==========================================

mae = mean_absolute_error(y_test, y_pred)

rmse = mean_squared_error(
    y_test,
    y_pred
) ** 0.5

r2 = r2_score(y_test, y_pred)


print("\n===== LINEAR REGRESSION RESULTS =====")

print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 3))

# ==========================================
# CROSS VALIDATION
# ==========================================

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    model,
    X,
    y,
    cv=kf,
    scoring="neg_mean_absolute_error"
)

cv_mae = -cv_scores

print("\n===== 5-FOLD CROSS VALIDATION =====")

print("MAE for each fold:",
      cv_mae.round(2))

print("Average MAE:",
      round(cv_mae.mean(), 2))

print("MAE standard deviation:",
      round(cv_mae.std(), 2))

# ==========================================
# 10. SHOW ACTUAL VS PREDICTED
# ==========================================

results = X_test.copy()

results["Actual Consumption"] = y_test.values
results["Predicted Consumption"] = y_pred.round(2)

results["Prediction Error"] = (
    results["Actual Consumption"]
    - results["Predicted Consumption"]
).round(2)

print("\n===== SAMPLE PREDICTIONS =====")
print(
    results[
        [
            "expected_students",
            "menu",
            "day_of_week",
            "weather",
            "temperature_c",
            "is_holiday",
            "is_exam_day",
            "special_event",
            "Actual Consumption",
            "Predicted Consumption",
            "Prediction Error"
        ]
    ].to_string(index=False))

Dataset loaded: (300, 18)
Training records: 240
Testing records: 60

===== LINEAR REGRESSION RESULTS =====
MAE : 14.11
RMSE: 16.84
R²  : 0.98

===== 5-FOLD CROSS VALIDATION =====
MAE for each fold: [14.11 13.07 12.76 12.39 11.19]
Average MAE: 12.7
MAE standard deviation: 0.95

===== SAMPLE PREDICTIONS =====
 expected_students          menu day_of_week weather  temperature_c  is_holiday  is_exam_day  special_event  Actual Consumption  Predicted Consumption  Prediction Error
               135   Idli Sambar    Thursday  Normal           27.9           1            0              0                 119                 113.79              5.21
               507    Rajma Rice    Thursday     Hot           36.6           0            0              0                 440                 426.27             13.73
               447      Dal Roti     Tuesday  Normal           30.4           0            1              0                 398                 401.28             -3.28
               

In [71]:
#  USING RANDOM FOREST REGRASSION

from sklearn.ensemble import RandomForestRegressor
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ==========================================
# 1. LOAD CLEANED DATASET
# ==========================================

df = pd.read_csv(
    "food_waste_management_dataset_cleaned.csv"
)

print("Dataset loaded:", df.shape)


# ==========================================
# 2. SELECT FEATURES
# ==========================================

features = [
    "expected_students",
    "menu",
    "day_of_week",
    "weather",
    "temperature_c",
    "is_holiday",
    "is_exam_day",
    "special_event"
]

target = "food_consumed_portions"

X = df[features]
y = df[target]


# ==========================================
# 3. IDENTIFY COLUMN TYPES
# ==========================================

categorical_features = [
    "menu",
    "day_of_week",
    "weather"
]

numerical_features = [
    "expected_students",
    "temperature_c",
    "is_holiday",
    "is_exam_day",
    "special_event"
]


# ==========================================
# 4. PREPROCESSING
# ==========================================

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


# ==========================================
# 5. CREATE MODEL PIPELINE
# ==========================================

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
         "regressor",
         RandomForestRegressor(
         n_estimators=200,
         max_depth=8,
         random_state=42
    )
)
    ]
)


# ==========================================
# 6. TRAIN / TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))


# ==========================================
# 7. TRAIN MODEL
# ==========================================

model.fit(X_train, y_train)


# ==========================================
# 8. MAKE PREDICTIONS
# ==========================================

y_pred = model.predict(X_test)


# ==========================================
# 9. EVALUATE MODEL
# ==========================================

mae = mean_absolute_error(y_test, y_pred)

rmse = mean_squared_error(
    y_test,
    y_pred
) ** 0.5

r2 = r2_score(y_test, y_pred)


print("\n===== RANDOM FOREST RESULTS =====")

print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 3))


# ==========================================
# 10. SHOW ACTUAL VS PREDICTED
# ==========================================

results = X_test.copy()

results["Actual Consumption"] = y_test.values
results["Predicted Consumption"] = y_pred.round(2)

results["Prediction Error"] = (
    results["Actual Consumption"]
    - results["Predicted Consumption"]
).round(2)

print(
    results[
        [
            "expected_students",
            "menu",
            "day_of_week",
            "weather",
            "temperature_c",
            "is_holiday",
            "is_exam_day",
            "special_event",
            "Actual Consumption",
            "Predicted Consumption",
            "Prediction Error"
        ]
    ].to_string(index=False))


Dataset loaded: (300, 18)
Training records: 240
Testing records: 60

===== RANDOM FOREST RESULTS =====
MAE : 16.54
RMSE: 21.21
R²  : 0.968
 expected_students          menu day_of_week weather  temperature_c  is_holiday  is_exam_day  special_event  Actual Consumption  Predicted Consumption  Prediction Error
               135   Idli Sambar    Thursday  Normal           27.9           1            0              0                 119                 113.29              5.71
               507    Rajma Rice    Thursday     Hot           36.6           0            0              0                 440                 444.53             -4.53
               447      Dal Roti     Tuesday  Normal           30.4           0            1              0                 398                 399.26             -1.26
               346   Idli Sambar    Saturday     Hot           36.6           0            0              0                 281                 289.48             -8.48
               3

In [72]:
#  USING RANDOM Gradient Boosting Regressor

from sklearn.ensemble import GradientBoostingRegressor
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ==========================================
# 1. LOAD CLEANED DATASET
# ==========================================

df = pd.read_csv(
    "food_waste_management_dataset_cleaned.csv"
)

print("Dataset loaded:", df.shape)


# ==========================================
# 2. SELECT FEATURES
# ==========================================

features = [
    "expected_students",
    "menu",
    "day_of_week",
    "weather",
    "temperature_c",
    "is_holiday",
    "is_exam_day",
    "special_event"
]

target = "food_consumed_portions"

X = df[features]
y = df[target]


# ==========================================
# 3. IDENTIFY COLUMN TYPES
# ==========================================

categorical_features = [
    "menu",
    "day_of_week",
    "weather"
]

numerical_features = [
    "expected_students",
    "temperature_c",
    "is_holiday",
    "is_exam_day",
    "special_event"
]


# ==========================================
# 4. PREPROCESSING
# ==========================================

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


# ==========================================
# 5. CREATE MODEL PIPELINE
# ==========================================

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
        "regressor",
        GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
)
    ]
)


# ==========================================
# 6. TRAIN / TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))


# ==========================================
# 7. TRAIN MODEL
# ==========================================

model.fit(X_train, y_train)


# ==========================================
# 8. MAKE PREDICTIONS
# ==========================================

y_pred = model.predict(X_test)


# ==========================================
# 9. EVALUATE MODEL
# ==========================================

mae = mean_absolute_error(y_test, y_pred)

rmse = mean_squared_error(
    y_test,
    y_pred
) ** 0.5

r2 = r2_score(y_test, y_pred)


print("\n===== GRADIENT BOOSTING RESULTS =====")

print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 3))


# ==========================================
# 10. SHOW ACTUAL VS PREDICTED
# ==========================================

results = X_test.copy()

results["Actual Consumption"] = y_test.values
results["Predicted Consumption"] = y_pred.round(2)

results["Prediction Error"] = (
    results["Actual Consumption"]
    - results["Predicted Consumption"]
).round(2)

print(
    results[
        [
            "expected_students",
            "menu",
            "day_of_week",
            "weather",
            "temperature_c",
            "is_holiday",
            "is_exam_day",
            "special_event",
            "Actual Consumption",
            "Predicted Consumption",
            "Prediction Error"
        ]
    ].to_string(index=False))


Dataset loaded: (300, 18)
Training records: 240
Testing records: 60

===== GRADIENT BOOSTING RESULTS =====
MAE : 16.06
RMSE: 19.53
R²  : 0.973
 expected_students          menu day_of_week weather  temperature_c  is_holiday  is_exam_day  special_event  Actual Consumption  Predicted Consumption  Prediction Error
               135   Idli Sambar    Thursday  Normal           27.9           1            0              0                 119                 112.88              6.12
               507    Rajma Rice    Thursday     Hot           36.6           0            0              0                 440                 434.38              5.62
               447      Dal Roti     Tuesday  Normal           30.4           0            1              0                 398                 407.45             -9.45
               346   Idli Sambar    Saturday     Hot           36.6           0            0              0                 281                 267.42             13.58
            